# Quantifying the statistical significance of an optimization

using the validation benchmark results to determine the statistical significance of the optimization results.

In [ ]:
import json
import pandas as pd

import numpy as np
from scipy.stats import ttest_rel
from scipy import stats

from cohens_d import cohens_d

import plotly.io as pio

In [ ]:
baseline_run_files = [
    "" # enter results file to analyse
]

comparing_run_files = [
    "" # enter results file to analyse
]

In [ ]:
baseline_data = []

for run_file in baseline_run_files:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                baseline_data.append(r)

baseline_df = pd.DataFrame(baseline_data)

baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
comparing_data = []

for run_file in comparing_run_files:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                comparing_data.append(r)

comparing_df = pd.DataFrame(comparing_data)

comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
metrics = [
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

ttest_results = []

results = []

for metric in metrics:
    baseline = baseline_df[metric].values
    comparing = comparing_df[metric].values

    ttest = ttest_rel(comparing, baseline, nan_policy='omit', alternative='two-sided')
    ttest_results.append(ttest)
    
    cohensd = cohens_d(comparing, baseline, paired=True, nan_policy='omit', alternative='two-sided')

    results.append({
        "metric": metric,
        "baseline_mean": np.nanmean(baseline),
        "baseline_error_margin": stats.sem(baseline, nan_policy='omit')*1.96,
        "comparing_mean": np.nanmean(comparing),
        "comparing_error_margin": stats.sem(comparing, nan_policy='omit')*1.96,
        "pvalue": ttest.pvalue,
        "cohensd": cohensd
    })

pd.DataFrame(results)

In [ ]:
# error probability for 15 t-Tests with alpha=0.05

err_prob = 1-pow((1-0.05),15)
err_prob

In [ ]:
significance_niveau = 0.05

def bonferroni_correction(significance, num_tests):
    return significance / num_tests

significance_niveau = bonferroni_correction(significance_niveau, 15)

significance_niveau

In [ ]:
import plotly.graph_objects as go

baseline_means = baseline_df[metrics].mean()
baseline_sems = baseline_df[metrics].sem()*1.96

comparing_means = comparing_df[metrics].mean()
comparing_sems = comparing_df[metrics].sem()*1.96

colors = ["#56D752" if (r.statistic > 0 and r.pvalue <= significance_niveau) 
            else "#FF7B52" if (r.statistic < 0 and r.pvalue <= significance_niveau) 
            else "black" for r in ttest_results]

baseline_text = [
    f" {np.round(baseline_mean, 2)} ±{f"{baseline_sems.iloc[i]}"[1:5]}"
    for i, baseline_mean in enumerate(baseline_means)    
]
comparing_text = [
    f" {np.round(comparing_mean, 2)} ±{f"{comparing_sems.iloc[i]}"[1:5]}"
    for i, comparing_mean in enumerate(comparing_means)    
]

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Baseline',
    x=metrics,
    y=baseline_means,
    error_y=dict(
        type='data', 
        array=baseline_sems, 
        visible=True,
        thickness=1,
        width=5
    ),
    marker_color='grey',
    text=baseline_text
))

fig.add_trace(go.Bar(
    name='Comparing',
    x=metrics,
    y=comparing_means,
    error_y=dict(
        type='data', 
        array=comparing_sems, 
        visible=True,
        thickness=1,
        width=5
    ),
    marker=dict(color=colors),
    text=comparing_text
))

fig.update_layout(
    # title='Validation results comparison per metric<br>(Mean ± SEM)',
    xaxis_title='Metric',
    yaxis_title='Score',
    barmode='group',
    template='plotly_white',
    showlegend=False,
    height=550,
    width=1000,
    font=dict(
        size=18,
    )
)

fig.show()
pio.write_image(fig, 'validation-per-metric.pdf')

In [ ]:
datasets = [
    "corporate QA", "fact checking", "general QA", "multi-hop QA", "multiple choice"
]

results = []
ttest_results = []

baseline_means = []
baseline_sems = []

comparing_means = []
comparing_sems = []

for dataset in datasets:
    baseline = baseline_df[baseline_df['dataset'] == dataset]
    comparing = comparing_df[comparing_df['dataset'] == dataset]

    baseline_total = []
    comparing_total = []

    for metric in metrics:
        baseline_total = [ *baseline_total,  *baseline[metric].values ]
        comparing_total = [ *comparing_total, *comparing[metric].values ]

    baseline_means.append(np.nanmean(baseline_total))
    baseline_sems.append(stats.sem(baseline_total, nan_policy='omit')*1.96)

    comparing_means.append(np.nanmean(comparing_total))
    comparing_sems.append(stats.sem(comparing_total, nan_policy='omit')*1.96)

    ttest = ttest_rel(comparing_total, baseline_total, nan_policy='omit', alternative='two-sided')
    ttest_results.append(ttest)

    cohensd = cohens_d(comparing_total, baseline_total, paired=True, nan_policy='omit', alternative='two-sided')

    results.append({
        "dataset": dataset,
        "baseline_mean": np.nanmean(baseline_total),
        "baseline_error_margin": stats.sem(baseline_total, nan_policy='omit')*1.96,
        "comparing_mean": np.nanmean(comparing_total),
        "comparing_error_margin": stats.sem(comparing_total, nan_policy='omit')*1.96,
        "pvalue": ttest.pvalue,
        "cohensd": cohensd
    })

pd.DataFrame(results)


In [ ]:
import plotly.graph_objects as go

colors = ["#56D752" if (r.statistic > 0 and r.pvalue <= significance_niveau) 
            else "#FF7B52" if (r.statistic < 0 and r.pvalue <= significance_niveau) 
            else "black" for r in ttest_results]

baseline_text = [
    f" {np.round(baseline_mean, 2)} ±{f"{baseline_sems[i]}"[1:5]}"
    for i, baseline_mean in enumerate(baseline_means)    
]
comparing_text = [
    f" {np.round(comparing_mean, 2)} ±{f"{comparing_sems[i]}"[1:5]}"
    for i, comparing_mean in enumerate(comparing_means)    
]

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Baseline',
    x=datasets,
    y=baseline_means,
    error_y=dict(
        type='data', 
        array=baseline_sems, 
        visible=True,
        thickness=1,
        width=5
    ),
    marker_color='grey',
    text=baseline_text
))

fig.add_trace(go.Bar(
    name='Comparing',
    x=datasets,
    y=comparing_means,
    error_y=dict(
        type='data', 
        array=comparing_sems, 
        visible=True,
        thickness=1,
        width=5
    ),
    marker=dict(color=colors),
    text=comparing_text
))

fig.update_layout(
    # title='Validation results comparison per dataset<br>(Mean ± SEM)',
    xaxis_title='Dataset',
    yaxis_title='Score',
    barmode='group',
    template='plotly_white',
    showlegend=False,
    height=500,
    width=600,
    font=dict(
        size=16,
    )
)

fig.show()
pio.write_image(fig, 'validation-per-dataset.pdf')

In [ ]:
baseline_total = []
comparing_total = []

for metric in metrics:
    baseline_total = [ *baseline_total,  *baseline_df[metric].values ]
    comparing_total = [ *comparing_total, *comparing_df[metric].values ]

baseline_mean = np.nanmean(baseline_total)
baseline_error_margin = stats.sem(baseline_total, nan_policy='omit')*1.96

comparing_mean = np.nanmean(comparing_total)
comparing_error_margin = stats.sem(comparing_total, nan_policy='omit')*1.96

results = {
    "baseline" : {
        "mean": baseline_mean,
        "sme": baseline_error_margin
    },
    "comparing": {
        "mean": comparing_mean,
        "sme": comparing_error_margin
    }
}

ttest = ttest_rel(comparing_total, baseline_total, nan_policy='omit', alternative='two-sided')
cohensd = cohens_d(comparing_total, baseline_total, paired=True, nan_policy='omit', alternative='two-sided')

results = [{
    "baseline_mean": baseline_mean,
    "baseline_error_margin": baseline_error_margin,
    "comparing_mean": comparing_mean,
    "comparing_error_margin": comparing_error_margin,
    "pvalue": ttest.pvalue,
    "cohensd": cohensd
}]

pd.DataFrame(results)


In [ ]:
detail = comparing_df.groupby('dataset')[metrics].mean()

text_matrix = detail.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

deatil_fig = go.Figure(data=go.Heatmap(
    z=detail,
    x=detail.columns,
    y=detail.index,
    text=text_matrix,
    zmin=0.0,
    zmax=1.0,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#FF7B52"), 
                (0.5, "#FFC44F"), (0.7, "#FFC44F"),
                (0.7, "#F1FF53"), (0.9, "#F1FF53"),
                (0.9, "#56D752"),  (1.00, "#56D752")]
))

deatil_fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    # title="Detaillierte Validierungsergebnisse der finalen Konfiguration",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)
    
deatil_fig.show()
pio.write_image(deatil_fig, 'validation-detail-scores.png', scale=5)

In [ ]:

baseline_detail = baseline_df.groupby('dataset')[metrics].mean()
comparing_detail = comparing_df.groupby('dataset')[metrics].mean()

heatmap_data = comparing_detail - baseline_detail

text_matrix = heatmap_data.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    text=text_matrix,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#F1FF53"), (1.00, "#56D752")],
))

fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    # title="Detaillierter Vergleich der Validierungsergebnisse der <br>finalen Konfiguration gegen die Baseline",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)

fig.show()
pio.write_image(fig, 'validation-detail-delta.png', scale=5)

In [ ]:
responses = comparing_df.loc[comparing_df['dataset'] == 'fact checking']['response']

total_fact_checking_tasks = 0
answers_with_bib = 0

for res in responses:
    total_fact_checking_tasks += 1
    if "<bib>" in res:
        answers_with_bib += 1

print(f"{np.round((answers_with_bib / total_fact_checking_tasks) * 100, 2)}% of fact checking tasks have a generated bibliography")